#  **Unstructured - PDF RAG** 



---

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import logging

# 로깅 레벨 설정 
logging.getLogger('pdfminer').setLevel(logging.ERROR)
logging.getLogger('unstructured').setLevel(logging.ERROR)

---

## **테슬라 10-K 리포트 PDF 기반 RAG 시스템 구축 (Unstructured & Langchain)**

- **테슬라 10-K 보고서** PDF를 `unstructured` 라이브러리로 파싱

- **Langchain**과 **ChromaDB**를 활용한 RAG 시스템 구축

- PDF 문서 기반 **지능형 질의응답** 시스템 구현 프로젝트

### 1. **문서 로드**

- **테슬라 10-K 보고서** PDF 파일을 **인터넷에서 다운로드**

- **SEC EDGAR 웹사이트** 또는 **테슬라 IR 페이지**에서 다운로드 가능

- 출처: https://ir.tesla.com/#quarterly-disclosure

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 경로 지정
file_path = "data/tsla-20241231-gen.pdf"

# PyPDFLoader 초기화
loader = PyPDFLoader(file_path)

# PDF 문서 로드
pypdf_docs = loader.load()

# 로드된 문서 개수 확인
print(len(pypdf_docs)) 

In [ ]:
# 첫 번째 페이지의 내용 출력
print(f"{pypdf_docs[0].page_content}\n")
print("-" * 100)

# 페이지 메타데이터 확인 
pprint(pypdf_docs[0].metadata)

### 2. **문서 파싱 (Unstructured)**

- **Unstructured 라이브러리**를 사용하여 **PDF 문서 파싱** 및 텍스트 추출을 수행

`(1) 문서 로드`

- LangChain에서 제공하는 UnstructuredLoader 사용   

In [ ]:
# 추출한 이미지 저장 폴더 
image_folder = "data/images/tesla_10k"
os.makedirs(image_folder, exist_ok=True)

In [ ]:
from langchain_unstructured import UnstructuredLoader
from unstructured.cleaners.core import (
    clean_extra_whitespace,
    replace_unicode_quotes,
    clean_non_ascii_chars,
    group_broken_paragraphs
)

# 로더 생성
loader = UnstructuredLoader(

    # PDF 파일 경로
    file_path,             

    # 파티셔닝 전략 설정
    strategy="hi_res",                  
    hi_res_model_name="yolox",          
    infer_table_structure=True,         
    languages=["eng"],           
    
    # 이미지 추출 설정
    extract_images_in_pdf=True,         
    extract_image_block_types=["Image", "Table"],
    extract_image_block_output_dir=image_folder,  

    # 후처리 설정
    post_processors=[
        clean_extra_whitespace,  # 불필요한 공백 제거
        replace_unicode_quotes,  # 유니코드 따옴표 제거
        clean_non_ascii_chars,   # 비 ASCII 문자 제거
        group_broken_paragraphs,  # 줄바꿈으로 분리된 문단 결합
        ], 
)


# 문서 출력
docs = []
for doc in loader.lazy_load():
    docs.append(doc)

# 문서 개수 확인
print(len(docs))

In [ ]:
# 문서 출력
for doc in docs[:5]:
    pprint(doc.page_content)
    print("-" * 100)
    pprint(doc.metadata)
    print("=" * 100)
    print()

In [ ]:
# 첫 번째 문서의 내용 출력
pprint(docs[0].page_content)

In [ ]:
# 메타데이터 확인
pprint(docs[0].metadata)

In [ ]:
# 문서 객체를 pickle로 저장
import pickle
with open("data/tesla_10k.pkl", "wb") as f:
    pickle.dump(docs, f)

In [ ]:
from langchain_core.documents import Document

# pickle로 저장된 문서 객체 로드
with open("data/tesla_10k.pkl", "rb") as f:
    pickled_docs = pickle.load(f)

# 문서 개수 확인
print(len(pickled_docs))

# 첫 번째 문서의 내용 출력
print(f"{pickled_docs[0].page_content}\n")

print("-" * 100)

# 첫 번째 문서의 메타데이터 확인
pprint(pickled_docs[0].metadata)

`(2) 표 데이터 변환`

- 테이블 형식의 데이터를 마크다운 텍스트로 변환하여 처리 

In [ ]:
# 문서 구성 요소의 유형 확인
category_counts = {}

for doc in docs:
    category = doc.metadata["category"]
    if category in category_counts:
        category_counts[category] += 1
    else:
        category_counts[category] = 1

# 카테고리별 문서 개수 출력
pprint(category_counts)

In [ ]:
# 카테고리 별로 문서 재정리 

new_docs = []
for doc in docs:

    # Table 카테고리의 문서를 마크다운 형식으로 변환
    if doc.metadata["category"] == "Table":
        # 판다스 데이터프레임으로 변환
        _df = pd.read_html(doc.metadata['text_as_html'])[0]

        # 마크다운 형식으로 변환
        _md = _df.to_markdown(index=False)

        # 새로운 문서 객체 생성
        new_docs.append(Document(page_content=_md, metadata=doc.metadata))

    # Header, Footer, Image 카테고리의 문서는 제외
    elif doc.metadata["category"] in ["Header", "Footer", "Image"]:
        continue

    # 나머지 문서는 그대로 추가
    else:
        new_docs.append(doc)
    
# 변환된 문서 개수 확인
print(len(new_docs))

### 3. **텍스트 청킹 (Langchain)**

- **텍스트 청킹**은 대규모 텍스트를 의미 있는 **작은 단위로 분할**하는 기술

- 주요 목적은 **벡터 데이터베이스 저장 및 검색 효율성 향상**

`(1) 적정 청크 크기`

In [ ]:
# 각 문서의 page_content 길이 확인

doc_lengths = [len(doc.page_content) for doc in new_docs]

# 문서 길이 시각화
import matplotlib.pyplot as plt

plt.hist(doc_lengths, bins=50)
plt.show()

In [ ]:
# 문서 타입별 개수 확인
category_counts = {}

for doc in new_docs:
    category = doc.metadata["category"]
    if category in category_counts:
        category_counts[category] += 1
    else:
        category_counts[category] = 1

# 카테고리별 문서 개수 출력
pprint(category_counts)

In [ ]:
# 문서 타입별로 문서 길이 확인
category_lengths = {}
for doc in new_docs:
    category = doc.metadata["category"]
    if category not in category_lengths:
        category_lengths[category] = []
    category_lengths[category].append(len(doc.page_content))

# 카테고리별 문서 길이 시각화
for category, lengths in category_lengths.items():
    plt.hist(lengths, bins=50)
    plt.title(f"Document Lengths for {category}")
    plt.xlabel("Length")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
# 문서 타입별 토큰 수 확인 (tiktoken tokenizer 사용)

import tiktoken

# tiktoken tokenizer 초기화
tokenizer = tiktoken.get_encoding("cl100k_base")

# 문서 타입별로 토큰 수 확인
category_tokens = {}
for doc in new_docs:
    category = doc.metadata["category"]
    if category not in category_tokens:
        category_tokens[category] = []
    category_tokens[category].append(len(tokenizer.encode(doc.page_content)))
    
# 카테고리별 토큰 수 시각화
for category, tokens in category_tokens.items():
    plt.hist(tokens, bins=50)
    plt.title(f"Document Tokens for {category}")
    plt.xlabel("Tokens")
    plt.ylabel("Frequency")
    plt.show()

`(2) 목차 항목 추출`

In [ ]:
# 3페이지(page_number) 목차 항목을 기준으로 그룹화 

toc_items = [ doc.page_content for doc in new_docs if doc.metadata["page_number"] == 3]
toc_items

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# 목차 항목 모델 정의
class Item(BaseModel):
    number: str = Field(description="목차 항목 번호 (예: 1, 1A, 1B, 2, 3 등)")
    title: str = Field(description="목차 항목 제목")

class Section(BaseModel):
    section: str = Field(description="목차 항목 그룹")
    items: list[Item] = Field(description="목차 항목 리스트")

# 목차 텍스트에서 항목을 추출하기 위한 프롬프트
TOC_PROMPT = """
You are a helpful assistant.
The user will provide you with a list of items.

Your task is to group these items into sections based on their content.

Please provide the output in JSON format.
The JSON should contain the following fields:
- section: The name of the section
- items: A list of items that belong to this section

The items are as follows:
{items}
"""

# 프롬프트 템플릿 생성
toc_prompt_template = PromptTemplate(
    input_variables=["items"],
    template=TOC_PROMPT
)

# LLM 정의 
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
llm_structured = llm.with_structured_output(Section)

# LCEL 체인 
toc_chain = toc_prompt_template | llm_structured

# 목차 항목을 기준으로 그룹화
toc_section = toc_chain.invoke({"items": "\n\n".join(toc_items)})

# 그룹화된 목차 항목 확인
pprint(toc_section)


In [ ]:
for item in toc_section.items:
    print(f"{item.number}: {item.title}")

In [ ]:
def get_item_header(item_number, item_title):
    """가능한 목차 항목 제목 변형을 반환하는 함수"""
    variations = [
        f"ITEM {item_number.upper()}. {item_title.upper()}",
    ]
    return variations

toc_based_docs = []
current_section = None

for i, doc in enumerate(new_docs):
    # 이 문서가 목차 항목에 해당하는지 확인
    new_section_found = False
    
    for item in toc_section.items:
        item_headers = get_item_header(item.number, item.title)
        
        # 목차 항목 제목 변형이 문서에 포함되어 있는지 확인
        if any(header in doc.page_content for header in item_headers):
            current_section = item.title
            new_section_found = True
            break

    # page_content에서 목차 항목 부분부터 추출
    if new_section_found:
        # 문서의 page_content에서 목차 항목 부분부터 추출
        start_index = min([doc.page_content.index(header) for header in item_headers if header in doc.page_content])
        doc.page_content = doc.page_content[start_index:]
    
    # 문서에 섹션 메타데이터 추가
    doc_copy = Document(
        page_content=doc.page_content,
        metadata={
            **doc.metadata,
            "section": current_section if current_section else "Unknown"
        }
    )
    toc_based_docs.append(doc_copy)

# 섹션별 문서 개수 확인
section_counts = {}
for doc in toc_based_docs:
    section = doc.metadata.get("section", "Unknown")
    section_counts[section] = section_counts.get(section, 0) + 1

print("Section distribution:")
for section, count in section_counts.items():
    print(f"{section}: {count} documents")

In [ ]:
# 목차 항목의 순서에 따라 섹션 정렬
section_order = {item.title: idx for idx, item in enumerate(toc_section.items)}

# "Unknown" 섹션을 마지막에 추가
section_order["Unknown"] = len(section_order)

# 섹션 순서에 따라 문서 정렬
toc_based_docs_sorted = sorted(
    toc_based_docs, 
    key=lambda x: section_order.get(x.metadata.get("section", "Unknown"), float('inf'))
)

# 섹션별 문서 개수 확인 (정렬된 문서)
section_counts_sorted = {}
for doc in toc_based_docs_sorted:
    section = doc.metadata.get("section", "Unknown")
    section_counts_sorted[section] = section_counts_sorted.get(section, 0) + 1

# 정렬된 섹션 분포 출력
print("\nSorted section distribution:")
for section in sorted(section_counts_sorted.keys(), key=lambda x: section_order.get(x, float('inf'))):
    print(f"{section}: {section_counts_sorted[section]} documents")

In [ ]:
# 문서 객체를 pickle로 저장
with open("data/tesla_10k_toc.pkl", "wb") as f:
    pickle.dump(toc_based_docs_sorted, f)

In [ ]:
# pickle로 저장된 문서 객체 로드
with open("data/tesla_10k_toc.pkl", "rb") as f:
    toc_based_docs_sorted = pickle.load(f)

# 문서 개수 확인
print(len(toc_based_docs_sorted))

`(3) 섹션별로 결합`

- 보고서 섹션별로 문서 객체를 결합하여 재구조화 

In [ ]:
# Unknown 제외 나머지 섹션별로 결합 

section_docs = {}

for doc in toc_based_docs_sorted:
    section = doc.metadata.get("section", "Unknown")
    if section == "Unknown":
        continue
    if section not in section_docs:
        section_docs[section] = []
    section_docs[section].append(doc)

# 섹션별로 문서 결합
section_docs_combined = {}
for section, docs in section_docs.items():
    combined_content = "\n\n".join([doc.page_content for doc in docs])
    combined_metadata = docs[0].metadata
    section_docs_combined[section] = Document(page_content=combined_content, metadata=combined_metadata)


# 결합된 문서 개수 확인
print(len(section_docs_combined))

In [ ]:
# 문서 저장
with open("data/tesla_10k_sections.pkl", "wb") as f:
    pickle.dump(section_docs_combined, f)

In [ ]:
# pickle로 저장된 문서 객체 로드
import pickle
with open("data/tesla_10k_sections.pkl", "rb") as f:
    section_docs_combined = pickle.load(f)

# 문서 개수 확인
print(len(section_docs_combined))

In [ ]:
# 첫 번째 문서의 내용 출력
print(f"{section_docs_combined['Business'].page_content}\n")

# 첫 번째 문서의 메타데이터 확인
pprint(section_docs_combined['Business'].metadata)

### 4. **Parent Document Retriever**

- 문서 검색 시 **작은 단위 분할**과 **문맥 유지** 사이의 균형이 중요함

- 작은 단위로 분할하면 **임베딩의 정확도**가 높아지나 문맥이 손실될 수 있음

- **ParentDocumentRetriever**는 작은 청크로 저장하고 검색 시 상위 문서를 반환하여 두 가지 목표를 달성함

- 효과적인 문서 검색을 위해 청크 크기와 문맥 보존 사이의 최적점을 찾는 것이 핵심

In [ ]:
# 각 섹션별로 문서 길이 확인
section_lengths = {section: len(doc.page_content) for section, doc in section_docs_combined.items()}

# 길이 순서로 정렬
section_lengths = dict(sorted(section_lengths.items(), key=lambda item: item[1]))

# 섹션별 문서 길이 시각화
plt.barh(section_lengths.keys(), section_lengths.values())
plt.xticks(rotation=90)
plt.xlabel("Section")
plt.ylabel("Length")
plt.title("Document Lengths by Section")
plt.show()

In [ ]:
import tiktoken

# tiktoken tokenizer 초기화
tokenizer = tiktoken.get_encoding("cl100k_base")

# 각 섹션별로 토큰 수 확인
section_tokens = {section: len(tokenizer.encode(doc.page_content)) for section, doc in section_docs_combined.items()}

# 길이 순서로 정렬
section_tokens = dict(sorted(section_tokens.items(), key=lambda item: item[1]))

# 섹션별 토큰 수 시각화
plt.barh(section_tokens.keys(), section_tokens.values())
plt.xticks(rotation=90)
plt.xlabel("Section")
plt.ylabel("Tokens")
plt.title("Document Tokens by Section")
plt.show()

In [ ]:
# 첫 번째 섹션의 메타데이터 확인
pprint(section_docs_combined['Business'].metadata)

In [ ]:
# 토큰 수 기준으로 3000개 이상이면 분할 (섹션 내의 순서 정보를 metadata에 추가)
from langchain_core.documents import Document
import tiktoken


tokenizer = tiktoken.get_encoding("cl100k_base")

section_docs_split = {}

for section, doc in section_docs_combined.items():

    filtered_metadata = {
        'element_id': doc.metadata['element_id'],
        'parent_id': doc.metadata['parent_id'] if 'parent_id' in doc.metadata else None,
        'source': doc.metadata['source'],
        'page_number': doc.metadata['page_number'],   
        'section': section        
    }

    tokens = len(tokenizer.encode(doc.page_content))
    if tokens > 3000:
        # 문서 분할
        split_docs = []
        for i in range(0, len(doc.page_content), 3000):
            split_doc = Document(
                page_content=doc.page_content[i:i+3000],
                metadata={
                    **filtered_metadata,
                    "order": i // 3000 + 1
                }
            )
            split_docs.append(split_doc)
        section_docs_split[section] = split_docs
    else:
        # 문서 그대로 추가
        doc.metadata = filtered_metadata
        doc.metadata["order"] = 1
        section_docs_split[section] = [doc]


# 분할된 문서 개수 확인
len([doc for docs in section_docs_split.values() for doc in docs])

In [ ]:
section_docs_split['Business'][0].metadata 

In [ ]:
# 문서 객체를 pickle로 저장
with open("data/tesla_10k_sections_split.pkl", "wb") as f:
    pickle.dump(section_docs_split, f)

In [ ]:
# pickle로 저장된 문서 객체 로드
with open("data/tesla_10k_sections_split.pkl", "rb") as f:
    section_docs_split = pickle.load(f)

# 문서 개수 확인
print(len([doc for docs in section_docs_split.values() for doc in docs]))

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import LocalFileStore
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os
import pickle

# 문서 저장소 경로 설정
storage_path = "./document_store"
os.makedirs(storage_path, exist_ok=True)

# 부모 문서 저장소 클래스 정의
class PickleFileStore(LocalFileStore):
    def mget(self, keys):
        """Get the values for the given keys."""
        return [pickle.loads(v) if v is not None else None 
                for v in super().mget(keys)]

    def mset(self, key_value_pairs):
        """Set the values for the given key-value pairs."""
        serialized_pairs = [
            (k, pickle.dumps(v)) for k, v in key_value_pairs
        ]
        super().mset(serialized_pairs)

# 부모 문서 저장소 초기화
store = PickleFileStore(storage_path)

# 부모 문서용 텍스트 스플리터 (섹션 단위로 큰 청크)
parent_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=2000,
    chunk_overlap=400,
    separators=["\n\n", "\n", " ", ""]
)

# 자식 문서용 텍스트 스플리터 (검색용 작은 청크)
child_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

# 벡터 스토어 초기화
vectorstore = Chroma(
    collection_name="tesla_10k_sections",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# ParentDocumentRetriever 설정
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 문서 추가
retriever.add_documents(
    [
        doc
        for docs in section_docs_split.values()
        for doc in docs
    ]
)

In [ ]:
# 저장된 문서 수 확인 
print(f"Total documents in vectorstore: {vectorstore._collection.count()}")

# 로컬 저장소 문서 수 확인
all_keys = list(store.yield_keys())
print(f"Total documents in store: {len(all_keys)}")


In [ ]:
# Test 검색
query = "What are the main risk factors?"
retrieved_docs = retriever.invoke(query)

print(f"Retrieved {len(retrieved_docs)} documents")
for doc in retrieved_docs:
    print(doc.page_content[:500])
    print("-" * 100)

In [ ]:
# 벡터 스토어 로드
vectorstore = Chroma(
    collection_name="tesla_10k_sections",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# 로컬 파일 저장소 로드
storage_path = "./document_store"
store = PickleFileStore(storage_path)

# ParentDocumentRetriever 설정
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# Test 검색
query = "What are the main risk factors?"

retrieved_docs = retriever.invoke(query)
print(f"Retrieved {len(retrieved_docs)} documents")

for doc in retrieved_docs:
    print(doc.page_content[:500])
    print("-" * 100)

In [ ]:
# 다른 쿼리 

query = "Where is Tesla's headquarters located?"

retrieved_docs = retriever.invoke(query)
print(f"Retrieved {len(retrieved_docs)} documents")

for doc in retrieved_docs:
    print(doc.page_content[:500])
    print("-" * 100)

In [ ]:
# vectorstore.delete_collection()

### 5. **다중 벡터 기반 검색(Multi Vector Retrieval)**

- **Multi-Vector Retriever**는 문서를 **여러 벡터**로 분할하여 저장하고 검색하는 시스템
    1. 벡터 저장소(Vectorstore): 임베딩 저장
    2. 문서 저장소(Docstore): 원본 문서 저장

- 예제: **요약 기반 검색 시스템** 구현

    - 각 섹션의 간결한 요약문을 생성하여 벡터 저장소에 보관하고 유사도 검색에 활용함
    - **영구 저장소**는 PickleFileStore와 Chroma를 사용하여 문서와 벡터 데이터를 세션 간 유지함
    - **검색 과정**은 요약문으로 의미론적 매칭을 수행하고 전체 원본 문서를 반환하여 완전한 문맥을 제공함
    - **RAG 파이프라인**은 ChatGPT를 활용하여 요약 생성과 최종 답변을 처리하며 명확한 응답 형식을 제공함

In [ ]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import LocalFileStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os
import pickle

# 로컬 파일 저장소 클래스 정의
class PickleFileStore(LocalFileStore):
    def mget(self, keys):
        return [pickle.loads(v) if v is not None else None 
                for v in super().mget(keys)]

    def mset(self, key_value_pairs):
        serialized_pairs = [(k, pickle.dumps(v)) for k, v in key_value_pairs]
        super().mset(serialized_pairs)

# 저장소 디렉토리 생성
os.makedirs("./summary_store", exist_ok=True)
os.makedirs("./chroma_db", exist_ok=True)

# 로컬 파일 저장소 초기화
doc_store = PickleFileStore("./summary_store")
vectorstore = Chroma(
    collection_name="tesla_10k_summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# MultiVectorRetriever 설정
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=doc_store,
    id_key="doc_id"   # 문서 ID 키 설정
)

# 각 문서에 대한 요약 생성
def generate_summary(text):
    summary_prompt = ChatPromptTemplate.from_template(
        "Summarize the following 10-K report section in 2-3 sentences:\n\n{text}"
    )
    model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
    summary_chain = summary_prompt | model | StrOutputParser()
    return summary_chain.invoke({"text": text})

# 요약된 문서 저장
summary_docs = []
original_docs = []
doc_ids = []

# 각 섹션별로 문서 결합
to_index_docs = [
        doc
        for docs in section_docs_split.values()
        for doc in docs
]

# 각 문서에 대한 요약 생성
for i, doc in enumerate(to_index_docs):
    doc_id = f"doc_{i}"
    summary = generate_summary(doc.page_content)
    
    # 요약 문서 생성
    summary_doc = Document(
        page_content=summary,
        metadata={"doc_id": doc_id, "section": doc.metadata["section"]}
    )
    summary_docs.append(summary_doc)
    
    # 원본 문서와 ID 저장
    original_docs.append(doc)
    doc_ids.append(doc_id)

# 벡터 스토어에 요약 문서 추가
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, original_docs)))

In [ ]:
# 로컬 파일 저장소 로드
doc_store = PickleFileStore("./summary_store")
vectorstore = Chroma(
    collection_name="tesla_10k_summaries",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory="./chroma_db"
)

# 문서 개수 확인
print(f"Total documents in vectorstore: {vectorstore._collection.count()}")
all_keys = list(doc_store.yield_keys())
print(f"Total documents in store: {len(all_keys)}")

In [ ]:
# MultiVectorRetriever 설정
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=doc_store,
    id_key="doc_id"
)

In [ ]:
# 문서 검색 
query = "What are the main risk factors?"

retrieved_docs = retriever.invoke(query)
print(f"Retrieved {len(retrieved_docs)} documents")

for doc in retrieved_docs:
    print(doc.page_content[:500])
    print("-" * 100)

In [ ]:
# RAG 체인 생성
rag_prompt = ChatPromptTemplate.from_template("""
Answer the following question based on the provided context from a 10-K report.
If you cannot find the answer in the context, say "제공된 문서에서 답을 찾을 수 없습니다."

<Context>
{context}
</Context>

<Question>
{question}
</Question>

<Answer>""")

model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | model
    | StrOutputParser()
)

# 질문 생성
question = "What are the main risk factors?"
answer = chain.invoke(question)
print(f"\nQuestion: {question}")
print(f"Answer: {answer}")

---

## **[실습] PDF 문서 기반 RAG 시스템 구축**

- 이전 코드를 기반으로 비정형 문서 전처리 기법을 적용하여 RAG 성능을 개선합니다. 

- 문서 파티셔닝, 재구조화 및 청킹, Multi Vector 검색기 등 개선요소를 찾아서 적용합니다. 


In [ ]:
# 여기에 코드를 작성하세요. 

In [ ]:
# ================================================================================
# Phase 9: 전체 시스템 테스트 및 성능 평가
# ================================================================================

import time

print("\nPhase 9: 전체 시스템 종합 테스트")
print("=" * 80)

# 다양한 유형의 테스트 쿼리
test_queries_comprehensive = [
    {
        "query": "What are Tesla's main risk factors mentioned in the report?",
        "category": "Risk Analysis",
        "expected_section": "Risk Factors"
    },
    {
        "query": "Where is Tesla's headquarters located and what are the main facilities?",
        "category": "Company Info",
        "expected_section": "Business"
    },
    {
        "query": "What are Tesla's revenue sources and business segments?",
        "category": "Business Model",
        "expected_section": "Business"
    },
    {
        "query": "What are the key financial metrics and performance indicators?",
        "category": "Financial",
        "expected_section": "Financial Statements"
    },
    {
        "query": "What legal proceedings is Tesla involved in?",
        "category": "Legal",
        "expected_section": "Legal Proceedings"
    }
]

# 각 RAG 시스템 비교
print("\n비교 테스트: 기본 vs 향상된 vs 최종 시스템")
print("=" * 80)

rag_systems = {
    "Basic (Parent only)": rag_chain_parent,
    "Enhanced (2-way Ensemble)": enhanced_rag_chain,
    "Final (3-way Ensemble)": final_rag_chain
}

# 테스트 쿼리 실행 (첫 2개만 테스트)
for i, test_case in enumerate(test_queries_comprehensive[:2], 1):
    query = test_case["query"]
    category = test_case["category"]
    
    print(f"\n{'='*80}")
    print(f"Test {i}: {category}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    
    for system_name, rag_chain in rag_systems.items():
        print(f"\n--- {system_name} ---")
        
        start_time = time.time()
        try:
            answer = rag_chain.invoke(query)
            elapsed = time.time() - start_time
            
            print(f"Response Time: {elapsed:.2f}s")
            print(f"Answer: {answer[:400]}...")
        except Exception as e:
            print(f"Error: {str(e)[:200]}")
        
        print("-" * 80)

# 시스템 통계 요약
print("\n\n시스템 통계 요약")
print("=" * 80)

print("\n1. 문서 처리 통계:")
print(f"  - 원본 문서 요소 수: {len(docs)}")
print(f"  - 정리된 문서 수: {len(new_docs)}")
print(f"  - 섹션 수: {len(section_docs_combined)}")
print(f"  - 최종 분할 문서 수: {len(docs_to_index)}")

print("\n2. 벡터 스토어 통계:")
print(f"  - Parent Retriever 벡터: {vectorstore_parent._collection.count()}")
print(f"  - Summary Retriever 벡터: {vectorstore_summary._collection.count()}")
print(f"  - Keyword Retriever 벡터: {vectorstore_keyword._collection.count()}")

print("\n3. 검색 전략:")
print("  - Parent Document Retriever:")
print(f"    * Child chunk: 400 tokens (검색용)")
print(f"    * Parent chunk: 1500 tokens (컨텍스트용)")
print("  - Summary-based Multi-Vector:")
print(f"    * 요약문으로 검색, 원본 반환")
print(f"    * Search k=6")
print("  - Keyword-based Multi-Vector:")
print(f"    * 키워드/질문으로 검색")
print(f"    * Search k=5")

print("\n4. Ensemble 가중치:")
print("  - Parent Retriever: 0.4")
print("  - Summary Retriever: 0.4")
print("  - Keyword Retriever: 0.2")

# 성능 개선 요약
print("\n\n성능 개선 요약")
print("=" * 80)

improvements = [
    {
        "feature": "문서 파싱",
        "before": "PyPDFLoader (기본)",
        "after": "UnstructuredLoader (hi_res, table extraction)",
        "benefit": "표 구조 인식, 메타데이터 풍부"
    },
    {
        "feature": "청킹 전략",
        "before": "고정 크기 청킹",
        "after": "적응형 청킹 (섹션별 최적화)",
        "benefit": "문맥 보존 및 검색 정밀도 향상"
    },
    {
        "feature": "검색 방식",
        "before": "단일 벡터 검색",
        "after": "3-way Ensemble (Parent + Summary + Keyword)",
        "benefit": "다각도 검색, 재현율 향상"
    },
    {
        "feature": "컨텍스트 구성",
        "before": "단순 나열",
        "after": "섹션별 그룹화, 출처 표시",
        "benefit": "가독성 및 추적성 향상"
    },
    {
        "feature": "프롬프트",
        "before": "기본 Q&A 프롬프트",
        "after": "구조화된 답변, 출처 인용 강제",
        "benefit": "답변 품질 및 신뢰성 향상"
    }
]

for i, improvement in enumerate(improvements, 1):
    print(f"\n{i}. {improvement['feature']}")
    print(f"  Before: {improvement['before']}")
    print(f"  After: {improvement['after']}")
    print(f"  Benefit: {improvement['benefit']}")

# 추가 개선 가능 영역
print("\n\n추가 개선 가능 영역")
print("=" * 80)

future_improvements = [
    "Cross-encoder를 활용한 재랭킹 (정확도 추가 향상)",
    "Hypothetical Document Embeddings (HyDE) 적용",
    "Query transformation/expansion (쿼리 다양화)",
    "Contextual compression (관련 없는 정보 필터링)",
    "Self-RAG (답변 품질 자가 평가 및 반복)",
    "캐싱 시스템 (반복 쿼리 성능 향상)",
    "A/B 테스팅 프레임워크 (정량적 평가)"
]

for i, item in enumerate(future_improvements, 1):
    print(f"  {i}. {item}")

print("\n" + "=" * 80)
print("Phase 9 완료: 전체 시스템 테스트 및 평가 완료")
print("=" * 80)

print("\n\n최종 요약:")
print("=" * 80)
print("Tesla 10-K 보고서 기반 향상된 RAG 시스템이 성공적으로 구축되었습니다.")
print("\n주요 구성 요소:")
print("1. Unstructured를 활용한 고급 문서 파싱")
print("2. TOC 기반 문서 재구조화 및 메타데이터 enrichment")
print("3. 적응형 청킹 전략 (섹션별 최적화)")
print("4. 3-way Ensemble Retrieval (Parent + Summary + Keyword)")
print("5. 구조화된 프롬프트 및 출처 인용")
print("\n이 시스템은 기본 RAG 대비 검색 품질, 답변 정확도, 사용자 경험 측면에서")
print("상당한 개선을 제공합니다.")
print("=" * 80)

In [ ]:
# ================================================================================
# Phase 7 & 8: 선택적 고도화 - 적응형 청킹 및 키워드 기반 Multi-Vector
# ================================================================================

print("\nPhase 7: 적응형 청킹 전략 구현")
print("=" * 80)

# 섹션별 특성 분석 함수
def analyze_section_characteristics(section_name, docs):
    """섹션의 특성을 분석하여 최적 청킹 전략 결정"""
    # 표 개수 계산
    table_count = sum(1 for doc in docs if doc.metadata.get("category") == "Table")
    
    # 평균 토큰 수 계산
    total_tokens = sum(len(tokenizer.encode(doc.page_content)) for doc in docs)
    avg_tokens = total_tokens / len(docs) if docs else 0
    
    # 전체 토큰 수
    total_text_tokens = len(tokenizer.encode("\n\n".join([doc.page_content for doc in docs])))
    
    return {
        "section": section_name,
        "doc_count": len(docs),
        "table_count": table_count,
        "avg_tokens": avg_tokens,
        "total_tokens": total_text_tokens,
        "has_tables": table_count > 0
    }

# 청킹 전략 결정 함수
def get_adaptive_chunk_strategy(characteristics):
    """섹션 특성에 따른 적응형 청킹 전략 반환"""
    section = characteristics["section"]
    has_tables = characteristics["has_tables"]
    avg_tokens = characteristics["avg_tokens"]
    
    # 재무 관련 섹션이나 표가 많은 경우: 작은 청크
    if has_tables or "financial" in section.lower() or "statement" in section.lower():
        return {
            "parent_size": 1000,
            "parent_overlap": 200,
            "child_size": 300,
            "child_overlap": 60,
            "reason": "Financial/Table-heavy section - smaller chunks for precision"
        }
    
    # 긴 서술형 섹션: 큰 청크
    elif avg_tokens > 1500 or "business" in section.lower() or "description" in section.lower():
        return {
            "parent_size": 2000,
            "parent_overlap": 400,
            "child_size": 600,
            "child_overlap": 120,
            "reason": "Narrative section - larger chunks for context"
        }
    
    # 기본 전략
    else:
        return {
            "parent_size": 1500,
            "parent_overlap": 300,
            "child_size": 400,
            "child_overlap": 80,
            "reason": "Standard chunking strategy"
        }

# 섹션별 특성 분석
print("섹션별 특성 분석:")
print("-" * 80)

section_characteristics = {}
for section, docs in section_docs.items():
    char = analyze_section_characteristics(section, docs)
    section_characteristics[section] = char
    strategy = get_adaptive_chunk_strategy(char)
    
    print(f"\n{section}:")
    print(f"  문서 수: {char['doc_count']}, 표 개수: {char['table_count']}")
    print(f"  평균 토큰: {char['avg_tokens']:.0f}, 전체 토큰: {char['total_tokens']}")
    print(f"  전략: {strategy['reason']}")
    print(f"  청크 크기: Parent={strategy['parent_size']}, Child={strategy['child_size']}")

print("\nPhase 8: 키워드 기반 Multi-Vector Retriever 추가")
print("=" * 80)

# 키워드 저장소 설정
keyword_store_path = "./keyword_store_improved"
os.makedirs(keyword_store_path, exist_ok=True)

keyword_doc_store = PickleFileStore(keyword_store_path)

# 키워드용 벡터 스토어
vectorstore_keyword = Chroma(
    collection_name="tesla_10k_keyword",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=chroma_path
)

# MultiVectorRetriever 설정 (키워드 기반)
keyword_retriever = MultiVectorRetriever(
    vectorstore=vectorstore_keyword,
    docstore=keyword_doc_store,
    id_key="doc_id",
    search_kwargs={"k": 5}
)

# 키워드/질문 생성 함수
def generate_keywords_and_questions(text, max_length=1500):
    """문서에서 핵심 키워드와 예상 질문 생성"""
    if len(text) > max_length:
        text = text[:max_length] + "..."
    
    keyword_prompt = ChatPromptTemplate.from_template(
        """Analyze the following text from Tesla's 10-K report and generate:
1. 5-7 key terms, phrases, or topics (comma-separated)
2. 2-3 questions that this text could answer

Text:
{text}

Output format:
Keywords: [your keywords here]
Questions:
- [question 1]
- [question 2]
- [question 3]
"""
    )
    
    keyword_chain = keyword_prompt | llm | StrOutputParser()
    return keyword_chain.invoke({"text": text})

print("키워드 및 질문 생성 중 (샘플링)...")

# 문서 샘플링 (비용 절감을 위해 일부만 처리)
sample_size = min(10, len(docs_to_index))
sampled_docs = docs_to_index[:sample_size]

keyword_docs = []
keyword_original_docs = []
keyword_doc_ids = []

for i, doc in enumerate(sampled_docs):
    doc_id = f"kw_doc_{i}"
    
    try:
        keywords_and_questions = generate_keywords_and_questions(doc.page_content)
    except:
        keywords_and_questions = f"Section: {doc.metadata.get('section', 'Unknown')}"
    
    # 키워드 문서 생성
    keyword_doc = Document(
        page_content=keywords_and_questions,
        metadata={
            "doc_id": doc_id,
            "section": doc.metadata.get("section", "Unknown"),
            "page_number": doc.metadata.get("page_number", "N/A")
        }
    )
    keyword_docs.append(keyword_doc)
    
    # 원본 문서 저장
    keyword_original_docs.append(doc)
    keyword_doc_ids.append(doc_id)
    
    if (i + 1) % 3 == 0:
        print(f"  진행률: {i + 1}/{sample_size}")

# 벡터 스토어에 추가
keyword_retriever.vectorstore.add_documents(keyword_docs)
keyword_retriever.docstore.mset(list(zip(keyword_doc_ids, keyword_original_docs)))

print(f"키워드 벡터 스토어 문서 수: {vectorstore_keyword._collection.count()}")
print(f"키워드 문서 저장소 문서 수: {len(list(keyword_doc_store.yield_keys()))}")

# 3-way Ensemble Retriever 구성
print("\n3-way Ensemble Retriever 구성:")
print("-" * 80)

three_way_ensemble = EnsembleRetriever(
    retrievers=[parent_retriever, summary_retriever, keyword_retriever],
    weights=[0.4, 0.4, 0.2],  # Parent와 Summary에 더 높은 가중치
    search_type="similarity"
)

print("3-way EnsembleRetriever 구성 완료")
print("  - Parent Document Retriever (가중치: 0.4)")
print("  - Summary-based Multi-Vector (가중치: 0.4)")
print("  - Keyword-based Multi-Vector (가중치: 0.2)")

# 최종 RAG 체인 (3-way ensemble)
final_rag_chain = (
    {
        "context": three_way_ensemble | format_docs_enhanced,
        "question": RunnablePassthrough()
    }
    | enhanced_rag_prompt
    | llm
    | StrOutputParser()
)

print("\nPhase 7 & 8 완료: 적응형 청킹 및 키워드 기반 Multi-Vector 완료")
print("=" * 80)

In [ ]:
# ================================================================================
# Phase 4, 5, 6: 우선 개선 - Multi-Vector, Ensemble, 향상된 프롬프트
# ================================================================================

from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.retrievers import EnsembleRetriever

print("\nPhase 4: Multi-Vector Retriever (요약 기반) 구축")
print("=" * 80)

# Multi-Vector용 저장소 설정
summary_store_path = "./summary_store_improved"
os.makedirs(summary_store_path, exist_ok=True)

# 요약 저장소 초기화
summary_doc_store = PickleFileStore(summary_store_path)

# 요약용 벡터 스토어 초기화
vectorstore_summary = Chroma(
    collection_name="tesla_10k_summary",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=chroma_path
)

# MultiVectorRetriever 설정
summary_retriever = MultiVectorRetriever(
    vectorstore=vectorstore_summary,
    docstore=summary_doc_store,
    id_key="doc_id",
    search_kwargs={"k": 6}  # 요약 검색은 더 많은 후보 확보
)

# 요약 생성 함수
def generate_summary(text, max_length=2000):
    """LLM을 사용하여 문서 요약 생성"""
    # 텍스트가 너무 길면 앞부분만 사용
    if len(text) > max_length:
        text = text[:max_length] + "..."
    
    summary_prompt = ChatPromptTemplate.from_template(
        """Summarize the following section from Tesla's 10-K report in 2-3 concise sentences.
Focus on the key points, numbers, and important information.

Text:
{text}

Summary:"""
    )
    
    summary_chain = summary_prompt | llm | StrOutputParser()
    return summary_chain.invoke({"text": text})

print("문서 요약 생성 중...")

# 요약 문서 생성 및 저장
summary_docs = []
original_docs = []
doc_ids = []

for i, doc in enumerate(docs_to_index):
    doc_id = f"doc_{i}"
    
    # 요약 생성
    try:
        summary = generate_summary(doc.page_content)
    except:
        # 요약 실패시 앞부분만 사용
        summary = doc.page_content[:300]
    
    # 요약 문서 생성 (벡터 스토어용)
    summary_doc = Document(
        page_content=summary,
        metadata={
            "doc_id": doc_id,
            "section": doc.metadata.get("section", "Unknown"),
            "page_number": doc.metadata.get("page_number", "N/A")
        }
    )
    summary_docs.append(summary_doc)
    
    # 원본 문서 저장 (docstore용)
    original_docs.append(doc)
    doc_ids.append(doc_id)
    
    if (i + 1) % 5 == 0:
        print(f"  진행률: {i + 1}/{len(docs_to_index)}")

# 벡터 스토어에 요약 추가
summary_retriever.vectorstore.add_documents(summary_docs)

# docstore에 원본 문서 추가
summary_retriever.docstore.mset(list(zip(doc_ids, original_docs)))

print(f"요약 벡터 스토어 문서 수: {vectorstore_summary._collection.count()}")
print(f"원본 문서 저장소 문서 수: {len(list(summary_doc_store.yield_keys()))}")

print("\nPhase 5: EnsembleRetriever로 통합")
print("=" * 80)

# Ensemble Retriever 구성 (Parent + Summary)
ensemble_retriever = EnsembleRetriever(
    retrievers=[parent_retriever, summary_retriever],
    weights=[0.5, 0.5],  # 동일한 가중치
    search_type="similarity"
)

print("EnsembleRetriever 구성 완료")
print("  - Parent Document Retriever (가중치: 0.5)")
print("  - Summary-based Multi-Vector Retriever (가중치: 0.5)")

print("\nPhase 6: 향상된 프롬프트 및 출처 표시")
print("=" * 80)

# 향상된 문서 포맷팅 함수
def format_docs_enhanced(docs):
    """섹션별로 그룹화하여 문서 포맷팅"""
    # 섹션별로 문서 그룹화
    sections = {}
    for doc in docs:
        section = doc.metadata.get('section', 'Unknown')
        if section not in sections:
            sections[section] = []
        sections[section].append(doc)
    
    # 포맷팅
    formatted = []
    for section_name, section_docs in sections.items():
        formatted.append(f"\n### Section: {section_name}")
        formatted.append("=" * 60)
        
        for i, doc in enumerate(section_docs, 1):
            page = doc.metadata.get('page_number', 'N/A')
            content = doc.page_content[:600] + "..." if len(doc.page_content) > 600 else doc.page_content
            
            formatted.append(f"\n[Excerpt {i} from page {page}]")
            formatted.append(content)
            formatted.append("-" * 40)
    
    return "\n".join(formatted)

# 향상된 RAG 프롬프트
enhanced_rag_prompt = ChatPromptTemplate.from_template("""You are an expert financial analyst analyzing Tesla's 10-K report.

Your task is to provide a comprehensive answer based on the context provided below.

Context (organized by sections):
{context}

Question: {question}

Instructions:
1. Provide a well-structured answer based on the context
2. ALWAYS cite your sources by mentioning the section name and page number
3. If information comes from multiple sections, organize your answer by section
4. If you cannot find sufficient information, clearly state what is missing
5. Use specific numbers, dates, and facts when available

Answer format:
- Start with a direct answer to the question
- Support with evidence from the context (cite section and page)
- Conclude with any important caveats or additional context

Answer:""")

# 향상된 RAG 체인 구성
enhanced_rag_chain = (
    {
        "context": ensemble_retriever | format_docs_enhanced,
        "question": RunnablePassthrough()
    }
    | enhanced_rag_prompt
    | llm
    | StrOutputParser()
)

# 테스트 쿼리
print("\n향상된 RAG 체인 테스트:")
print("-" * 80)

test_queries = [
    "What are Tesla's main risk factors?",
    "Where is Tesla's headquarters located?",
    "What are Tesla's main business segments?"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    print("-" * 80)
    answer = enhanced_rag_chain.invoke(query)
    print(answer)
    print("\n" + "=" * 80)

print("\nPhase 4, 5, 6 완료: Multi-Vector Retriever, Ensemble, 향상된 프롬프트 완료")
print("=" * 80)

In [ ]:
# ================================================================================
# Phase 2 & 3: 필수 구현 - Parent Document Retriever 및 기본 RAG 체인
# ================================================================================

from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import LocalFileStore
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough

print("\nPhase 2: Parent Document Retriever 구축")
print("=" * 80)

# 저장소 경로 설정
storage_path = "./document_store_improved"
chroma_path = "./chroma_db_improved"
os.makedirs(storage_path, exist_ok=True)
os.makedirs(chroma_path, exist_ok=True)

# PickleFileStore 클래스 정의
class PickleFileStore(LocalFileStore):
    """Document를 pickle로 직렬화하여 저장하는 스토어"""
    def mget(self, keys):
        return [pickle.loads(v) if v is not None else None 
                for v in super().mget(keys)]

    def mset(self, key_value_pairs):
        serialized_pairs = [(k, pickle.dumps(v)) for k, v in key_value_pairs]
        super().mset(serialized_pairs)

# 부모 문서 저장소 초기화
store = PickleFileStore(storage_path)

# 부모 문서용 텍스트 스플리터 (큰 청크로 context 유지)
parent_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1500,      # 부모 문서 크기 (context 풍부)
    chunk_overlap=300,    # 중복 영역
    separators=["\n\n", "\n", ". ", " ", ""]
)

# 자식 문서용 텍스트 스플리터 (작은 청크로 정밀 검색)
child_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400,       # 자식 문서 크기 (검색 정밀도)
    chunk_overlap=80,     # 중복 영역
    separators=["\n\n", "\n", ". ", " ", ""]
)

# 벡터 스토어 초기화
vectorstore_parent = Chroma(
    collection_name="tesla_10k_parent",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"),
    persist_directory=chroma_path
)

# ParentDocumentRetriever 설정
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore_parent,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

# 문서 추가 (섹션별 분할된 문서)
docs_to_index = [doc for docs in section_docs_split.values() for doc in docs]
print(f"인덱싱할 문서 수: {len(docs_to_index)}")

parent_retriever.add_documents(docs_to_index)

# 저장 확인
print(f"벡터 스토어 문서 수: {vectorstore_parent._collection.count()}")
print(f"부모 문서 저장소 문서 수: {len(list(store.yield_keys()))}")

print("\nPhase 3: 기본 RAG 체인 구성")
print("=" * 80)

# 문서 포맷팅 함수
def format_docs(docs):
    """검색된 문서를 컨텍스트 형식으로 포맷팅"""
    formatted = []
    for i, doc in enumerate(docs, 1):
        section = doc.metadata.get('section', 'Unknown')
        page = doc.metadata.get('page_number', 'N/A')
        content = doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content
        
        formatted.append(f"[Document {i}]")
        formatted.append(f"Section: {section} | Page: {page}")
        formatted.append(f"{content}")
        formatted.append("-" * 40)
    
    return "\n".join(formatted)

# RAG 프롬프트 템플릿
rag_prompt = ChatPromptTemplate.from_template("""You are a financial analyst expert. Answer the question based on the provided context from Tesla's 10-K report.

If you cannot find the answer in the context, say "I cannot find the answer in the provided documents."

Context:
{context}

Question: {question}

Instructions:
- Provide a clear and concise answer
- Cite the section and page number when possible
- If the information spans multiple sections, mention all relevant sections

Answer:""")

# LLM 설정
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# RAG 체인 구성 (LCEL)
rag_chain_parent = (
    {
        "context": parent_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# 테스트 쿼리
print("\nRAG 체인 테스트:")
print("-" * 80)
test_query = "What are Tesla's main risk factors?"
print(f"Query: {test_query}")
print(f"\nAnswer:")
answer = rag_chain_parent.invoke(test_query)
print(answer)

print("\nPhase 2 & 3 완료: Parent Document Retriever 및 기본 RAG 체인 완료")
print("=" * 80)

In [ ]:
# ================================================================================
# Phase 1: 필수 구현 - 문서 파싱 및 TOC 기반 재구조화
# ================================================================================

import os
import pickle
import pandas as pd
import tiktoken
from pprint import pprint
from langchain_core.documents import Document
from langchain_unstructured import UnstructuredLoader
from unstructured.cleaners.core import (
    clean_extra_whitespace,
    replace_unicode_quotes,
    clean_non_ascii_chars,
    group_broken_paragraphs
)
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

print("Step 1: 문서 로드 및 파싱")
print("-" * 80)

# 문서 파일 경로
file_path = "data/tsla-20241231-gen.pdf"

# 이미지 저장 폴더 생성
image_folder = "data/images/tesla_10k_improved"
os.makedirs(image_folder, exist_ok=True)

# UnstructuredLoader로 고급 파싱
loader = UnstructuredLoader(
    file_path,
    strategy="hi_res",
    hi_res_model_name="yolox",
    infer_table_structure=True,
    languages=["eng"],
    extract_images_in_pdf=True,
    extract_image_block_types=["Image", "Table"],
    extract_image_block_output_dir=image_folder,
    post_processors=[
        clean_extra_whitespace,
        replace_unicode_quotes,
        clean_non_ascii_chars,
        group_broken_paragraphs,
    ],
)

# 문서 로드
docs = []
for doc in loader.lazy_load():
    docs.append(doc)

print(f"로드된 문서 요소 수: {len(docs)}")

# 카테고리별 문서 재정리
print("\nStep 2: 카테고리별 문서 정리 및 표 변환")
print("-" * 80)

new_docs = []
for doc in docs:
    category = doc.metadata.get("category", "Unknown")
    
    # Table을 마크다운으로 변환
    if category == "Table":
        try:
            df = pd.read_html(doc.metadata['text_as_html'])[0]
            md = df.to_markdown(index=False)
            new_docs.append(Document(page_content=md, metadata=doc.metadata))
        except:
            # HTML 파싱 실패시 원본 텍스트 사용
            new_docs.append(doc)
    
    # Header, Footer, Image 제외
    elif category in ["Header", "Footer", "Image"]:
        continue
    
    # 나머지 문서는 그대로 추가
    else:
        new_docs.append(doc)

print(f"정리된 문서 수: {len(new_docs)}")

# 카테고리별 개수 확인
category_counts = {}
for doc in new_docs:
    cat = doc.metadata.get("category", "Unknown")
    category_counts[cat] = category_counts.get(cat, 0) + 1
print("카테고리별 분포:", category_counts)

# TOC 추출
print("\nStep 3: TOC(목차) 추출 및 섹션 분류")
print("-" * 80)

# 3페이지에서 목차 항목 추출
toc_items = [doc.page_content for doc in new_docs if doc.metadata.get("page_number") == 3]

if toc_items:
    # TOC 항목 모델 정의
    class Item(BaseModel):
        number: str = Field(description="목차 항목 번호")
        title: str = Field(description="목차 항목 제목")

    class Section(BaseModel):
        section: str = Field(description="목차 항목 그룹")
        items: list[Item] = Field(description="목차 항목 리스트")

    # LLM으로 TOC 구조화
    toc_prompt = PromptTemplate(
        input_variables=["items"],
        template="""You are a helpful assistant.
Extract the table of contents items from the following text.
Group them into sections and provide the output in JSON format.

Items:
{items}"""
    )
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    llm_structured = llm.with_structured_output(Section)
    toc_chain = toc_prompt | llm_structured
    
    toc_section = toc_chain.invoke({"items": "\n\n".join(toc_items)})
    print(f"추출된 섹션 수: {len(toc_section.items)}")
    
    # 각 문서에 섹션 메타데이터 추가
    def get_item_header(item_number, item_title):
        return [f"ITEM {item_number.upper()}. {item_title.upper()}"]
    
    toc_based_docs = []
    current_section = None
    
    for doc in new_docs:
        new_section_found = False
        
        for item in toc_section.items:
            item_headers = get_item_header(item.number, item.title)
            if any(header in doc.page_content for header in item_headers):
                current_section = item.title
                new_section_found = True
                
                # 목차 항목 부분부터 추출
                start_index = min([doc.page_content.index(h) for h in item_headers if h in doc.page_content])
                doc.page_content = doc.page_content[start_index:]
                break
        
        # 섹션 메타데이터 추가
        doc_copy = Document(
            page_content=doc.page_content,
            metadata={
                **doc.metadata,
                "section": current_section if current_section else "Unknown"
            }
        )
        toc_based_docs.append(doc_copy)
    
    # 섹션 순서대로 정렬
    section_order = {item.title: idx for idx, item in enumerate(toc_section.items)}
    section_order["Unknown"] = len(section_order)
    
    toc_based_docs_sorted = sorted(
        toc_based_docs,
        key=lambda x: section_order.get(x.metadata.get("section", "Unknown"), float('inf'))
    )
    
    print("섹션별 문서 분포:")
    section_counts = {}
    for doc in toc_based_docs_sorted:
        sec = doc.metadata.get("section", "Unknown")
        section_counts[sec] = section_counts.get(sec, 0) + 1
    for sec, cnt in section_counts.items():
        print(f"  {sec}: {cnt}")
else:
    print("목차를 찾을 수 없어 원본 문서 사용")
    toc_based_docs_sorted = new_docs
    toc_section = None

# 섹션별 결합
print("\nStep 4: 섹션별 문서 결합")
print("-" * 80)

section_docs = {}
for doc in toc_based_docs_sorted:
    section = doc.metadata.get("section", "Unknown")
    if section == "Unknown":
        continue
    if section not in section_docs:
        section_docs[section] = []
    section_docs[section].append(doc)

# 섹션별로 문서 결합
section_docs_combined = {}
for section, docs in section_docs.items():
    combined_content = "\n\n".join([doc.page_content for doc in docs])
    combined_metadata = docs[0].metadata
    section_docs_combined[section] = Document(
        page_content=combined_content,
        metadata=combined_metadata
    )

print(f"결합된 섹션 수: {len(section_docs_combined)}")

# 토큰 수 기준으로 분할 (3000 토큰 이상)
print("\nStep 5: 토큰 기준 문서 분할")
print("-" * 80)

tokenizer = tiktoken.get_encoding("cl100k_base")
section_docs_split = {}

for section, doc in section_docs_combined.items():
    filtered_metadata = {
        'element_id': doc.metadata.get('element_id'),
        'parent_id': doc.metadata.get('parent_id'),
        'source': doc.metadata.get('source'),
        'page_number': doc.metadata.get('page_number'),
        'section': section
    }
    
    tokens = len(tokenizer.encode(doc.page_content))
    
    if tokens > 3000:
        # 문서 분할
        split_docs = []
        for i in range(0, len(doc.page_content), 3000):
            split_doc = Document(
                page_content=doc.page_content[i:i+3000],
                metadata={**filtered_metadata, "order": i // 3000 + 1}
            )
            split_docs.append(split_doc)
        section_docs_split[section] = split_docs
    else:
        # 문서 그대로 추가
        doc.metadata = filtered_metadata
        doc.metadata["order"] = 1
        section_docs_split[section] = [doc]

total_docs = sum(len(docs) for docs in section_docs_split.values())
print(f"분할된 총 문서 수: {total_docs}")

print("\nPhase 1 완료: 문서 파싱 및 재구조화 완료")
print("=" * 80)